# Stage 7-3. Regression 전체 실험 — MLP + CNN 비교

이 노트북은 MNIST digit을 연속값으로 예측하는 회귀 과제의 전체 실험 파이프라인을 실습한다.

| 항목 | 값 |
|---|---|
| 과제 | digit label을 연속값으로 예측 |
| target 변환 | `label / 9.0` → shape `(1,)`, 범위 `[0, 1]` |
| output_dim | 1 |
| loss | mse |
| metric | r2_score |
| prediction_mode | `round(clip(logit × 9, 0, 9))` |

**학습 목표**
1. 회귀 과제의 target 변환, MSE loss, R² score metric, 후처리를 확인한다.
2. MLP vs CNN 학습 곡선(MSE/R²)과 예측 결과를 비교한다.
3. 저장된 checkpoint로 평가를 재현한다.
4. 3종 task(multiclass, binary, regression)의 설계 결정을 최종 정리한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt
import os

try:
    import cupy
    _BACKEND = f"CuPy {cupy.__version__}"
except ImportError:
    _BACKEND = "NumPy (CuPy 미설치 — fallback)"

print(f"백엔드: {_BACKEND}")

from src.core.experiment import Experiment
from src.core import checkpoints
from src.core.evaluator import Evaluator
from src.data.mnist import MnistDataset, load_mnist
from src.data.dataloader import DataLoader
from src.models.mlp import MLP
from src.models.cnn import CNN
from src.task import get_task_spec
from src.nn.activations import sigmoid

DATASET_DIR = "/mnt/d/datasets/mnist"
OUTPUT_DIR  = "../../outputs/regression"
TASK        = "regression"

print(f"\ntask: {TASK}")
spec = get_task_spec(TASK)
print(f"  output_dim     : {spec['output_dim']}")
print(f"  prediction_mode: {spec['prediction_mode']}")

## 1. Regression task 규약 확인

In [ ]:
ds = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
_, orig_labels = load_mnist("test", dataset_dir=DATASET_DIR)

print("[ Regression target 변환 확인 (처음 10개 샘플) ]")
print(f"  {'원본 digit':>12} {'target (÷9)':>12} {'복원 (×9)':>10}")
print("  " + "-" * 38)
for i in range(10):
    _, target = ds[i]
    digit  = int(orig_labels[i])
    t_val  = float(target[0])
    t_back = t_val * 9.0
    print(f"  {digit:>12} {t_val:>12.4f} {t_back:>10.1f}")

print()
print(f"target shape : {ds[0][1].shape}")
print(f"target dtype : {ds[0][1].dtype}")
print(f"target 변환  : label / 9.0  → [0.0, 1.0] 범위로 정규화")
print(f"예측 후처리  : round(clip(logit × 9, 0, 9)) → digit 정수")

In [ ]:
# target 분포 시각화
targets = np.array([float(ds[i][1][0]) * 9 for i in range(len(ds))])
counts  = np.bincount(targets.astype(int), minlength=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
fig.suptitle("Regression — target 분포 (test split)", fontsize=11, fontweight="bold")

# digit 분포
ax = axes[0]
bars = ax.bar(range(10), counts, color=plt.cm.tab10.colors, edgecolor="gray", alpha=0.85)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f"{cnt}", ha="center", fontsize=9)
ax.set_title("원본 digit 분포")
ax.set_xlabel("digit (0–9)")
ax.set_ylabel("샘플 수")
ax.set_xticks(range(10))
ax.grid(axis="y", alpha=0.3)

# 정규화된 target 분포
ax = axes[1]
all_targets = np.array([float(ds[i][1][0]) for i in range(len(ds))])
ax.hist(all_targets, bins=20, color="steelblue", edgecolor="gray", alpha=0.85)
ax.set_title("정규화 target 분포 (label/9.0)")
ax.set_xlabel("target value [0, 1]")
ax.set_ylabel("샘플 수")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 2. MLP 학습

In [ ]:
config_mlp = {
    "dataset_dir": DATASET_DIR,
    "task":        TASK,
    "model":       "mlp",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  10,
    "lr":          0.001,   # regression은 낮은 lr 사용
}

exp_mlp = Experiment(config_mlp)
print(f"MLP 파라미터 수: {sum(p.size for p in exp_mlp.model.params):,}")
print("학습 시작 (10 epochs, lr=0.001)...")
logs_mlp = exp_mlp.run()

print()
print(f"{'epoch':>5} | {'train loss(MSE)':>15} {'train R²':>10} | {'test loss':>10} {'test R²':>10}")
print("-" * 60)
for log in logs_mlp:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>15.6f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.6f} {log['test']['metric']:>10.4f}")

## 3. CNN 학습

In [ ]:
config_cnn = {
    "dataset_dir": DATASET_DIR,
    "task":        TASK,
    "model":       "cnn",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  10,
    "lr":          0.001,
}

exp_cnn = Experiment(config_cnn)
print(f"CNN 파라미터 수: {sum(p.size for p in exp_cnn.model.params):,}")
print("학습 시작 (10 epochs, lr=0.001)...")
logs_cnn = exp_cnn.run()

print()
print(f"{'epoch':>5} | {'train loss(MSE)':>15} {'train R²':>10} | {'test loss':>10} {'test R²':>10}")
print("-" * 60)
for log in logs_cnn:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>15.6f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.6f} {log['test']['metric']:>10.4f}")

## 4. 학습 곡선 비교

In [ ]:
epochs = list(range(1, 11))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Regression — MLP vs CNN (10 epochs, lr=0.001, batch=64)",
             fontsize=12, fontweight="bold")

for ax, key, title in zip(axes, ["loss", "metric"], ["MSE Loss", "R² Score"]):
    ax.plot(epochs, [l["train"][key] for l in logs_mlp], color="steelblue", linewidth=2,
            marker="o", label="MLP train")
    ax.plot(epochs, [l["test"][key]  for l in logs_mlp], color="steelblue", linewidth=2,
            marker="o", linestyle="--", label="MLP test")
    ax.plot(epochs, [l["train"][key] for l in logs_cnn], color="#E87B4C",   linewidth=2,
            marker="s", label="CNN train")
    ax.plot(epochs, [l["test"][key]  for l in logs_cnn], color="#E87B4C",   linewidth=2,
            marker="s", linestyle="--", label="CNN test")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("epoch")
    ax.set_ylabel(title.lower())
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# R² = 1이 이상값, 0은 baseline (평균 예측)
axes[1].axhline(0, color="gray", linestyle=":", linewidth=1.5, label="baseline (R²=0)")
axes[1].axhline(1, color="green", linestyle=":", linewidth=1.5, alpha=0.5, label="perfect (R²=1)")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Prediction 후처리 확인

In [ ]:
# round_clip 후처리 확인
test_ds = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
_, orig_test = load_mnist("test", dataset_dir=DATASET_DIR)
N = 8
imgs = np.stack([test_ds[i][0] for i in range(N)])

result = exp_mlp.predictor.predict(imgs)
logits = result["logits"]
preds  = result["predictions"]

print("[ Regression 예측 후처리 — round_clip ]")
print(f"  {'샘플':>4} {'원본 digit':>10} {'logit':>8} {'logit×9':>9} {'clip(0-9)':>10} {'round':>7}")
print("  " + "-" * 55)
for i in range(N):
    digit   = int(orig_test[i])
    logit   = float(logits[i, 0])
    scaled  = logit * 9.0
    clipped = np.clip(scaled, 0, 9)
    rounded = int(round(clipped))
    correct = "✓" if digit == rounded else "✗"
    print(f"  {i:>4} {digit:>10} {logit:>8.3f} {scaled:>9.3f} {clipped:>10.3f} {rounded:>7} {correct}")

print()
print("규칙: round(clip(logit × 9, 0, 9)) → digit 정수")
print("      logit이 음수여도 clip으로 0 이하 방지, 1 이상이어도 9 이하로 제한")

In [ ]:
# 예측 grid
N_SHOW = 16
imgs   = np.stack([test_ds[i][0] for i in range(N_SHOW)])
trues  = np.array([int(orig_test[i]) for i in range(N_SHOW)])

preds_mlp = exp_mlp.predictor.predict(imgs)["predictions"].flatten().astype(int)
preds_cnn = exp_cnn.predictor.predict(imgs)["predictions"].flatten().astype(int)

fig, axes = plt.subplots(2, 8, figsize=(14, 5))
fig.suptitle("Regression 예측 grid — MLP(위) vs CNN(아래) (10 epochs)",
             fontsize=11, fontweight="bold")

for col in range(8):
    for row, (preds, label) in enumerate([(preds_mlp, "MLP"), (preds_cnn, "CNN")]):
        ax = axes[row][col]
        ax.imshow(imgs[col].reshape(28, 28), cmap="gray")
        t, p = trues[col], preds[col]
        color = "green" if t == p else ("orange" if abs(t - p) <= 1 else "red")
        ax.set_title(f"T:{t} P:{p}", fontsize=7, color=color)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.show()

exact_mlp = (trues == preds_mlp).mean()
exact_cnn = (trues == preds_cnn).mean()
close_mlp = (np.abs(trues - preds_mlp) <= 1).mean()
close_cnn = (np.abs(trues - preds_cnn) <= 1).mean()
print(f"MLP — 정확 일치: {exact_mlp:.0%}, ±1 이내: {close_mlp:.0%}")
print(f"CNN — 정확 일치: {exact_cnn:.0%}, ±1 이내: {close_cnn:.0%}")

## 6. 정량 비교 및 checkpoint 재평가

In [ ]:
mlp_params = sum(p.size for p in exp_mlp.model.params)
cnn_params = sum(p.size for p in exp_cnn.model.params)
mlp_final  = logs_mlp[-1]["test"]
cnn_final  = logs_cnn[-1]["test"]

print("[ Regression 최종 결과 비교 ]")
print()
print(f"{'항목':<22} {'MLP':>12} {'CNN':>12}")
print("-" * 48)
print(f"{'파라미터 수':<22} {mlp_params:>12,} {cnn_params:>12,}")
print(f"{'test MSE loss':<22} {mlp_final['loss']:>12.6f} {cnn_final['loss']:>12.6f}")
print(f"{'test R² score':<22} {mlp_final['metric']:>12.4f} {cnn_final['metric']:>12.4f}")

print()
print("[ Phase 7.2 기준 결과 (저장된 checkpoint 평가) ]")
print(f"  MLP: loss=0.0408, R²=0.5984")
print(f"  CNN: loss=0.0703, R²=0.3104")
print()
print("  ※ Regression에서는 MLP가 CNN보다 높은 R² score를 기록했다.")
print("    CNN Regression은 학습 초기 수치 불안정 가능성이 있다.")

# 저장된 checkpoint 재평가
test_ds     = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
spec        = get_task_spec(TASK)

print()
print("[ 저장된 checkpoint 평가 ]")
for model_name, ModelClass in [("mlp", MLP), ("cnn", CNN)]:
    ck_path = os.path.join(OUTPUT_DIR, model_name, "model")
    if os.path.isfile(ck_path + ".npz"):
        model = ModelClass(task=TASK, seed=42)
        checkpoints.load(model, ck_path)
        evaluator = Evaluator(model, spec)
        result = evaluator.evaluate(test_loader)
        print(f"  {model_name.upper()}: loss={result['loss']:.6f}, "
              f"R²={result['metric']:.4f}, samples={result['num_samples']}")
    else:
        print(f"  {model_name.upper()}: checkpoint 없음")

## 7. 저장된 결과 이미지 확인

In [ ]:
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("outputs/regression/ — 저장된 결과 이미지", fontsize=12, fontweight="bold")

for row, model_name in enumerate(["mlp", "cnn"]):
    for col, fname in enumerate(["training_log.png", "predictions.png"]):
        ax = axes[row][col]
        fpath = os.path.join(OUTPUT_DIR, model_name, fname)
        if os.path.isfile(fpath):
            ax.imshow(np.asarray(Image.open(fpath)))
            ax.set_title(f"{model_name.upper()} — {fname}", fontsize=9)
        else:
            ax.text(0.5, 0.5, "파일 없음", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{model_name.upper()} — {fname}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

## 8. 3종 task 최종 비교

In [ ]:
# Phase 7.2 기준 결과 종합
results_ref = {
    "multiclass": {"mlp": {"loss": 0.4499, "metric": 0.8839}, "cnn": {"loss": 0.2106, "metric": 0.9345}},
    "binary":     {"mlp": {"loss": 0.2923, "metric": 0.8814}, "cnn": {"loss": 0.1780, "metric": 0.9347}},
    "regression": {"mlp": {"loss": 0.0408, "metric": 0.5984}, "cnn": {"loss": 0.0703, "metric": 0.3104}},
}
metric_names = {"multiclass": "accuracy", "binary": "accuracy", "regression": "R²"}

print("[ Phase 7.2 기준 — 전체 실험 결과 요약 ]")
print()
print(f"{'task':<14} {'model':<6} {'loss':>10} {'metric':>10} {'metric name':<14}")
print("-" * 58)
for task, models in results_ref.items():
    for model, res in models.items():
        print(f"{task:<14} {model:<6} {res['loss']:>10.4f} {res['metric']:>10.4f} {metric_names[task]:<14}")

In [ ]:
# 3종 task 비교 시각화
tasks  = ["multiclass", "binary", "regression"]
labels = ["Multiclass\n(accuracy)", "Binary\n(accuracy)", "Regression\n(R²)"]

mlp_metrics = [results_ref[t]["mlp"]["metric"] for t in tasks]
cnn_metrics = [results_ref[t]["cnn"]["metric"] for t in tasks]

x = np.arange(len(tasks))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_mlp = ax.bar(x - w/2, mlp_metrics, w, label="MLP", color="steelblue",  alpha=0.85, edgecolor="gray")
bars_cnn = ax.bar(x + w/2, cnn_metrics, w, label="CNN", color="#E87B4C",    alpha=0.85, edgecolor="gray")

for bar, val in zip(bars_mlp, mlp_metrics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", fontsize=9, color="steelblue", fontweight="bold")
for bar, val in zip(bars_cnn, cnn_metrics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", fontsize=9, color="#E87B4C",   fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("metric 값")
ax.set_title("3종 task — MLP vs CNN 최종 metric 비교 (Phase 7.2 기준)",
             fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="random baseline")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. CLI 실행 명령

In [ ]:
print("[ Regression 실험 CLI 실행 명령 ]")
print()
for model, env in [("mlp", "numpy_py311"), ("cnn", "cupy_py311_cuda121")]:
    print(f"# {model.upper()} ({env})")
    print(f"PYTHONPATH=. conda run -n {env} python scripts/train.py \\\ ")
    print(f"    --task regression --model {model} \\\ ")
    print(f"    --checkpoint outputs/regression/{model}/model.npz")
    print()
    print(f"PYTHONPATH=. conda run -n {env} python scripts/evaluate.py \\\ ")
    print(f"    --task regression --model {model} \\\ ")
    print(f"    --checkpoint outputs/regression/{model}/model.npz")
    print()

## 10. 정리 — 프로젝트 전체 설계 결정

**Regression 과제 규약**

| 항목 | 값 |
|---|---|
| target 변환 | `label / 9.0` → [0, 1] 정규화 |
| output_dim | 1 |
| loss | MSE |
| metric | R² score |
| prediction_mode | `round(clip(logit × 9, 0, 9))` |

**3종 task 최종 비교 (Phase 7.2 기준)**

| task | 모델 | loss | metric | 비고 |
|---|---|---|---|---|
| Multiclass | MLP | 0.4499 | acc=0.8839 | |
| Multiclass | CNN | 0.2106 | acc=0.9345 | CNN 우위 |
| Binary | MLP | 0.2923 | acc=0.8814 | |
| Binary | CNN | 0.1780 | acc=0.9347 | CNN 우위 |
| Regression | MLP | 0.0408 | R²=0.5984 | MLP 우위 |
| Regression | CNN | 0.0703 | R²=0.3104 | |

**후속 프레임워크 연계 핵심 인터페이스**

```python
# 동일한 3줄로 PyTorch / TensorFlow / JAX에서도 사용 가능
config = {"task": "multiclass", "model": "mlp", ...}
exp = Experiment(config)
logs = exp.run()
```

| 유지할 인터페이스 | 설명 |
|---|---|
| `config` 키 이름 | `task`, `model`, `seed`, `batch_size`, `num_epochs`, `lr` |
| `Trainer.fit()` 반환 | `{"loss": ..., "metric": ..., "num_samples": ...}` |
| `Evaluator.evaluate()` 반환 | 동일 |
| `Predictor.predict()` 반환 | `{"logits": ..., "predictions": ...}` |
| CLI 인자 이름 | `--task`, `--model`, `--checkpoint`, `--output_dir` |